# Regressão simbólica guiada por LLM para OTFTs (Org1, OVSED)

Adaptação de Taskin, Xie & Lazebnik (Sci Rep 2026): perda $L = w_1 e + w_2 s + w_3 c$, com
$c$ = penalidade de plausibilidade física atribuída por LLM. Equações-alvo: escada N1–N4
ancorada no modelo OVSED com parâmetros da Org1.

| Condição experimental | Guia do LLM | $w_3$ |
|---|---|---|
| A | nenhum (RS pura, linha de base) | 0 |
| B | prompt genérico (física clássica, artigo) | 0,1 |
| C | prompt de física de dispositivo (OTFT) | 0,1 |

Premissas: `references/premissas_cientificas.md` (H-OTFT-01…15).

## 0. Ambiente e parâmetros

In [1]:
# ---- PARÂMETROS EDITÁVEIS -------------------------------------------------------------
ORCAMENTO = "fumaca"            # "fumaca" (teste de encadeamento) | "completo" (best_config)
NIVEIS = ["N1", "N2", "N3", "N4"]
MOTORES = ["pysr", "deap", "gplearn"]
CONDICOES = ["A", "B", "C"]
RUIDOS = [0.0]                  # proposta §4.5: [0.0, 0.01, 0.05, 0.10]
SEMENTES = [0]
N_CURVAS, N_PONTOS = 20, 50     # curvas por nível, pontos por curva
BACKEND_LLM = "ollama"          # "nenhum" | "ollama" (mistral:7b-instruct, temperatura 0)
# Células B/C (LLM no laço): subconjunto da matriz; em fumaça, orçamento mínimo de DEAP.
NIVEIS_LLM, MOTORES_LLM = ["N2"], ["deap"]
ORCAMENTO_LLM_FUMACA = {"sr.population": 8, "sr.generations": 1, "sr.early_stopping.enabled": False}
# ---------------------------------------------------------------------------------------

In [2]:
import json, sys, time, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(RAIZ / "src"))
warnings.filterwarnings("ignore")

from otft_sr import ovsed
from otft_sr.contexto import contexto_fisico, criar_avaliador, ollama_disponivel
from otft_sr.dados import construir_datasets
from otft_sr.experimento import CRITERIO, PESOS_BASELINE, PESOS_LLM, controle_positivo, executar_condicao
from otft_sr.niveis import N5_DOCUMENTACAO, erro_vs_ovsed, escada, nivel

FUMACA = ORCAMENTO == "fumaca"
DIR_RESULTADOS = RAIZ / "results"
DIR_RESULTADOS.mkdir(exist_ok=True)
print(f"python: {sys.executable}")
print(f"orçamento: {ORCAMENTO}; pesos A = {tuple(round(w, 3) for w in PESOS_BASELINE)}, B/C = {PESOS_LLM}")
if FUMACA:
    print("ATENÇÃO: execução de FUMAÇA — valida o encadeamento; os números NÃO são resultados.")

python: /home/rsb6/Desktop/projeto cleber /physics-llm-sr/.venv/bin/python
orçamento: fumaca; pesos A = (0.997, 0.003, 0.0), B/C = (0.987166831194472, 0.002961500493583416, 0.00987166831194472)
ATENÇÃO: execução de FUMAÇA — valida o encadeamento; os números NÃO são resultados.


## 1. Modelo OVSED e escada de equações

OVSED (canal p em módulo): $\theta = (V_{GS} - V_{th0} - \delta V_{DS})/(n V_T)$,
$q_{tot} = \mathrm{softplus}(\theta)$, $I_D = I_{leak} + W J_{th} q_{tot}^{l} F_{sat}$, com
$R_S$ por ponto fixo. $V_{th0} = 8{,}57$ V (H-OTFT-02).

In [3]:
p = ovsed.carregar_parametros_org1()
curvas = ovsed.carregar_curvas_org1()
print("Parâmetros Org1:", p.como_dict())
print(f"RMSE log10|I_D| OVSED vs Org1 medida: V_th0 = 8,57 V -> {ovsed.erro_log_medido(p):.3f} déc; "
      f"V_th0 = 18,57 V -> {ovsed.erro_log_medido(ovsed.carregar_parametros_org1(V_th0=None)):.3f} déc")

linhas = []
for n in escada():
    e = erro_vs_ovsed(n)
    linhas.append({"nível": n.id, "nome": n.nome, "expressão de referência": n.expressao,
                   "I_D": n.unidades["I_D"], "região": n.regiao.descricao,
                   "erro rel. médio vs OVSED": round(e["erro_rel_medio"], 3),
                   "RMSE log10 vs OVSED (déc)": round(e["rmse_log10"], 3)})
tabela_niveis = pd.DataFrame(linhas).set_index("nível")
display(tabela_niveis)
for n in escada():
    print(f"{n.id}: {n.derivacao}")
print("N5:", N5_DOCUMENTACAO)

Parâmetros Org1: {'V_th0': 8.57, 'delta': 0.0123, 'n': 108.0, 'l': 3.18, 'Lambda': 1490.0, 'V_crit': 32.3, 'J_th': 0.097, 'R_S': 1.0, 'W': 0.1, 'I_leak': 6.5e-11}


RMSE log10|I_D| OVSED vs Org1 medida: V_th0 = 8,57 V -> 0.125 déc; V_th0 = 18,57 V -> 0.885 déc


,nome,expressão de referência,I_D,região,erro rel. médio vs OVSED,RMSE log10 vs OVSED (déc)
nível,,,,,,
N1,"Saturação quadrática (lei de Shockley, control...",0.008293 * (V_GS - 20.24) ** 2,uA,saturação: curvas de transferência com V_DS em...,0.092,0.049
N2,Lei de potência OVSED (mobilidade dependente d...,4.119e-05 * (V_GS - 8.57) ** 3.18,uA,acumulação forte em saturação: transferência c...,0.239,0.125
N3,Regime linear quadrático clássico (triodo),0.01173 * (2 * (V_GS - 25.52) * V_DS - V_DS ** 2),uA,regime linear (triodo): curvas de saída com V_...,0.100,0.054
N4,Corrente sublimiar exponencial,0.0006386 * exp(1.423 * V_GS),pA,sublimiar profundo (I_leak removido): transfer...,0.022,0.011


N1: K e V_Th por mínimo erro relativo quadrático contra o OVSED completo (Org1) na região de saturação; mesma ordem de grandeza de I_D da Org1 em saturação.
N2: V_th0 e l da Org1 fixos; K = W J_th F_sat,ref/(n V_T)^l com F_sat,ref por mínimo erro relativo contra o OVSED completo (F_sat,ref = 0.1031; F_sat do OVSED na região varia de 0.074 a 0.172).
N3: K e V_Th por mínimo erro relativo quadrático contra o OVSED completo no triodo (região dependente de V_Th, ajuste autoconsistente).
N4: Regressão de log10 I_D (OVSED sem I_leak) em V_GS: S = 1.618 V/déc. Referências: ln10 n V_T/l = 1.974 V/déc (só J_free) e ln10 n V_T/(l+1) = 1.502 V/déc (assíntota, F_sat ~ q_tot no sublimiar).
N5: N5 (saturação com R_S): I_D = K (V_GS - V_Th - I_D R_S)^(gamma+2) é implícita em I_D; a AST e os motores (DEAP/gplearn/PySR) só representam y = f(X) explícita. Fica como trabalho futuro (ex.: oferecer I_D como entrada e buscar o resíduo, ou RS sobre a inversa V_GS(I_D, V_DS)). Para a Org1, I_D R_S <= ~0,1 V (I

In [4]:
med = {c.nome: c for c in curvas}
N1, N2, N3, N4 = escada()
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))

c50 = med["org1_50VDS"]; vg = np.linspace(0.3, 50, 300)
ax[0].semilogy(c50.V_GS, c50.I_D_uA, "k.", label="Org1 medida (V_DS = 50 V)")
ax[0].semilogy(vg, ovsed.corrente_ovsed_uA(vg, 50, p), "k-", lw=1, label="OVSED completo")
for n, cor in ((N1, "C0"), (N2, "C1")):
    v = np.linspace(*n.regiao.V_GS, 100)
    ax[0].semilogy(v, n.gerador(v, np.full_like(v, 50.0)), cor, lw=2, label=f"{n.id}")
c2 = med["org1_2VDS"]
ax[0].set(xlabel="V_GS (V)", ylabel="I_D (µA)", title="Transferência, saturação: N1, N2")
ax[0].set_ylim(1e-5, 30); ax[0].legend(fontsize=8)

for vgs, cor in ((30, "C2"), (40, "C3"), (50, "C4")):
    c = med[f"org1_{vgs}VGS"]; vd = np.linspace(0, 50, 200)
    ax[1].plot(c.V_DS, c.I_D_uA, ".", color=cor, ms=4, label=f"Org1 medida V_GS = {vgs} V")
    ax[1].plot(vd, ovsed.corrente_ovsed_uA(vgs, vd, p), "-", color=cor, lw=1)
    vdt = np.linspace(0, vgs - N3.constantes["V_Th"], 50)
    ax[1].plot(vdt, N3.gerador(np.full_like(vdt, vgs), vdt), "--", color=cor, lw=2)
ax[1].set(xlabel="V_DS (V)", ylabel="I_D (µA)", title="Saída: OVSED (—) e N3 no triodo (--)")
ax[1].legend(fontsize=7)

vg = np.linspace(0, 20, 200)
ax[2].semilogy(c2.V_GS, c2.I_D_uA * 1e6, "k.", label="Org1 medida (V_DS = 2 V)")
ax[2].semilogy(vg, ovsed.corrente_ovsed_uA(vg, 2, p) * 1e6, "k-", lw=1, label="OVSED completo")
ax[2].semilogy(vg, ovsed.corrente_ovsed_uA(vg, 2, p, com_leak=False) * 1e6, "k:", label="OVSED sem I_leak")
v = np.linspace(*N4.regiao.V_GS, 50)
ax[2].semilogy(v, N4.gerador(v, np.full_like(v, 2.0)), "C5", lw=2, label="N4")
ax[2].set(xlabel="V_GS (V)", ylabel="I_D (pA)", title="Sublimiar: N4 (I_leak removido)")
ax[2].legend(fontsize=8)
fig.tight_layout(); fig.savefig(DIR_RESULTADOS / "fig_niveis_vs_ovsed_org1.png", dpi=150)
plt.show()

## 2. Dados

Grade por curvas (transferência a $V_{DS}$ fixo ou saída a $V_{GS}$ fixo), pontos sorteados
na região do nível, partição treino/validação/teste por curva (60/20/20 %), ruído gaussiano
multiplicativo $I_{obs} = I_D (1 + \sigma \mathcal{N}(0,1))$.

In [5]:
resumo = []
for nid in NIVEIS:
    ds = construir_datasets(nivel(nid), RUIDOS[-1], SEMENTES[0], N_CURVAS, N_PONTOS)
    ids = {k: set(np.unique(d.simulation_ids)) for k, d in ds.items()}
    assert not (ids["treino"] & ids["teste"]) and not (ids["treino"] & ids["validacao"])
    assert not (ids["validacao"] & ids["teste"])
    resumo.append({"nível": nid, "tipo de curva": ds["treino"].metadata["tipo_curva"],
                   **{f"curvas {k}": len(v) for k, v in ids.items()},
                   **{f"pontos {k}": d.y_observed.size for k, d in ds.items()},
                   "σ": RUIDOS[-1]})
display(pd.DataFrame(resumo).set_index("nível"))

fig, ax = plt.subplots(1, len(NIVEIS), figsize=(4 * len(NIVEIS), 3.2))
for a, nid in zip(np.atleast_1d(ax), NIVEIS):
    ds = construir_datasets(nivel(nid), RUIDOS[-1], SEMENTES[0], N_CURVAS, N_PONTOS)
    for k, m in (("treino", "."), ("validacao", "x"), ("teste", "+")):
        d = ds[k]
        a.scatter(d.X[:, 0], d.X[:, 1], s=4, marker=m, label=k)
    a.set(title=nid, xlabel="V_GS (V)", ylabel="V_DS (V)")
np.atleast_1d(ax)[0].legend(fontsize=7)
fig.tight_layout(); plt.show()

,tipo de curva,curvas treino,curvas validacao,curvas teste,pontos treino,pontos validacao,pontos teste,σ
nível,,,,,,,,
N1,transferencia,12,4,4,600,200,200,0.0
N2,transferencia,12,4,4,600,200,200,0.0
N3,saida,12,4,4,600,200,200,0.0
N4,transferencia,12,4,4,600,200,200,0.0


## 3. Controle positivo do pipeline

A expressão de referência, avaliada pelo mesmo objetivo e pelas mesmas métricas da busca,
deve dar $1 - R^2 \approx 0$, $e \approx 0$ e distância de árvore 0.

In [6]:
cp = pd.DataFrame([controle_positivo(nivel(nid)) for nid in NIVEIS]).set_index("nivel")
display(cp[["expressao", "n_nos", "e_normalizado", "one_minus_r2", "rmse_log10",
            "tree_distance", "tree_distance_algebraic"]])
assert (cp["one_minus_r2"] < 1e-12).all() and (cp["tree_distance"] == 0).all()
assert (cp["tree_distance_algebraic"] == 0).all()
print("controle positivo: OK")

,expressao,n_nos,e_normalizado,one_minus_r2,rmse_log10,tree_distance,tree_distance_algebraic
nivel,,,,,,,
N1,0.008293 * (V_GS - 20.24) ** 2.0,7,0.0,0.0,0.0,0.0,0.0
N2,4.119e-05 * (V_GS - 8.57) ** 3.18,7,0.0,0.0,0.0,0.0,0.0
N3,0.01173 * (2.0 * (V_GS - 25.52) * V_DS - V_DS ...,13,0.0,0.0,0.0,0.0,0.0
N4,0.0006386 * exp(1.423 * V_GS),6,0.0,0.0,0.0,0.0,0.0


controle positivo: OK


## 4. Linha de base sem LLM (condição A)

In [7]:
resultados = []
if "A" in CONDICOES:
    for nid in NIVEIS:
        for motor in MOTORES:
            for sigma in RUIDOS:
                for sem in SEMENTES:
                    r = executar_condicao(nivel(nid), motor, "A", sigma, sem, ORCAMENTO,
                                          n_curvas=N_CURVAS, n_pontos=N_PONTOS)
                    resultados.append(r)
                    print(f"A {nid} {motor:7s} σ={sigma:<4} s={sem} [{r['status']}] "
                          f"{r.get('expressao', r.get('erro'))!s:.70} | 1-R²={r.get('one_minus_r2', float('nan')):.2e} "
                          f"| {r['classe']} | {r['tempo_s']:.1f}s")

A N1 pysr    σ=0.0  s=0 [ok] 2.835675738077578 | 1-R²=1.02e+00 | não converge | 99.2s
A N1 deap    σ=0.0  s=0 [ok] log(V_GS) | 1-R²=8.68e-01 | não converge | 0.1s


A N1 gplearn σ=0.0  s=0 [ok] log((V_DS + V_DS) / (V_GS - V_DS + log(V_DS) - cos(cos(V_GS + (-0.9619 | 1-R²=7.45e-01 | não converge | 0.2s


A N2 pysr    σ=0.0  s=0 [ok] 1.8494138470825143 | 1-R²=1.00e+00 | não converge | 3.0s
A N2 deap    σ=0.0  s=0 [ok] V_DS / V_DS | 1-R²=1.29e+00 | não converge | 0.0s


A N2 gplearn σ=0.0  s=0 [ok] log((V_DS + V_DS) / (V_GS - V_DS)) | 1-R²=6.95e-01 | não converge | 0.3s
A N3 pysr    σ=0.0  s=0 [ok] 2.006235814654019 | 1-R²=3.52e+00 | não converge | 0.1s
A N3 deap    σ=0.0  s=0 [ok] log((V_DS + V_DS) / sin(V_DS)) | 1-R²=8.25e+00 | não converge | 0.0s


A N3 gplearn σ=0.0  s=0 [ok] log(V_DS) | 1-R²=8.84e-01 | não converge | 0.2s


A N4 pysr    σ=0.0  s=0 [ok] 0.0324618284635074 | 1-R²=1.01e+00 | não converge | 0.2s
A N4 deap    σ=0.0  s=0 [ok] 9.538057085207448 ** (-2.0409035384443452) | 1-R²=1.25e+00 | não converge | 0.0s


A N4 gplearn σ=0.0  s=0 [ok] V_GS - V_GS | 1-R²=1.58e+00 | não converge | 0.2s


## 5. Condições B e C (LLM no laço)

Variante de prompt E (variáveis + descrição do experimento, sem a equação-alvo). B usa
`prompts/generico`, C usa `prompts/dispositivo`; o contexto físico é o mesmo (H-OTFT-09).

In [8]:
condicoes_llm = [c for c in CONDICOES if c in ("B", "C")]
motivo = None
if not condicoes_llm:
    motivo = "nenhuma condição com LLM selecionada"
elif BACKEND_LLM == "nenhum":
    motivo = "BACKEND_LLM = 'nenhum'"
elif not ollama_disponivel():
    motivo = "Ollama indisponível ou sem mistral:7b-instruct"
if motivo:
    print(f"Condições B/C PULADAS: {motivo}.")
else:
    orc_llm = ORCAMENTO_LLM_FUMACA if FUMACA else ORCAMENTO
    for cond in condicoes_llm:
        avaliador = criar_avaliador(cond)
        for nid in NIVEIS_LLM:
            for motor in MOTORES_LLM:
                for sigma in RUIDOS:
                    for sem in SEMENTES:
                        r = executar_condicao(nivel(nid), motor, cond, sigma, sem, orc_llm,
                                              avaliador=avaliador, n_curvas=N_CURVAS, n_pontos=N_PONTOS)
                        r["orcamento"] = "fumaca" if FUMACA else r["orcamento"]
                        resultados.append(r)
                        print(f"{cond} {nid} {motor:7s} σ={sigma:<4} s={sem} [{r['status']}] "
                              f"{r.get('expressao', r.get('erro'))!s:.60} | 1-R²={r.get('one_minus_r2', float('nan')):.2e} "
                              f"| chamadas LLM={r['chamadas_llm']} ({r['tempo_llm_s']:.0f}s) | {r['tempo_s']:.1f}s")

B N2 deap    σ=0.0  s=0 [ok] log(exp(V_DS * (-3.3317580673250013))) | 1-R²=2.30e+00 | chamadas LLM=9 (103s) | 102.9s


C N2 deap    σ=0.0  s=0 [ok] log(exp(V_DS * (-3.3317580673250013))) | 1-R²=2.30e+00 | chamadas LLM=9 (52s) | 52.3s


## 6. Resultados

Critério de recuperação simbólica: $1 - R^2 \le 10^{-3}$ (teste limpo) e
$\min$(distância primária, equivalente, algébrica) $\le 0{,}05$, ou mesma forma funcional
por manipulação simbólica (sympy, H-OTFT-07b).

In [9]:
df = pd.DataFrame(resultados)
if FUMACA:
    print("FUMAÇA — tabela apenas demonstra o formato; NÃO é resultado.")
cols = ["nivel", "motor", "condicao", "ruido", "semente", "status", "expressao", "n_nos",
        "one_minus_r2", "rmse_log10", "tree_distance", "tree_distance_algebraic",
        "forma_funcional_sympy", "classe",
        "tempo_s", "chamadas_llm", "tempo_llm_s"]
display(df.reindex(columns=cols))
if len(df):
    ok = df[df["status"] == "ok"]
    resumo = ok.groupby(["nivel", "motor", "condicao"]).agg(
        recupera=("classe", lambda s: float(np.mean(s.str.startswith("recupera estrutura")))),
        one_minus_r2_mediana=("one_minus_r2", "median"),
        rmse_log10_mediana=("rmse_log10", "median"),
        dist_algebrica_mediana=("tree_distance_algebraic", "median"),
        nos_mediana=("n_nos", "median"),
        tempo_s=("tempo_s", "median"),
        chamadas_llm=("chamadas_llm", "sum"),
    )
    display(resumo)

FUMAÇA — tabela apenas demonstra o formato; NÃO é resultado.


,nivel,motor,condicao,ruido,semente,status,expressao,n_nos,one_minus_r2,rmse_log10,tree_distance,tree_distance_algebraic,forma_funcional_sympy,classe,tempo_s,chamadas_llm,tempo_llm_s
0,N1,pysr,A,0.0,0,ok,2.835675738077578,1,1.020656,0.474479,1.0,1.0,False,não converge,99.187349,0,0.000000
1,N1,deap,A,0.0,0,ok,log(V_GS),2,0.868459,0.491110,1.0,1.0,False,não converge,0.107167,0,0.000000
2,N1,gplearn,A,0.0,0,ok,log((V_DS + V_DS) / (V_GS - V_DS + log(V_DS) -...,35,0.744881,0.383453,1.0,1.0,False,não converge,0.234971,0,0.000000
3,N2,pysr,A,0.0,0,ok,1.8494138470825143,1,1.000653,0.575566,1.0,1.0,False,não converge,2.954868,0,0.000000
4,N2,deap,A,0.0,0,ok,V_DS / V_DS,3,1.289586,0.525873,1.0,1.0,False,não converge,0.029232,0,0.000000
5,N2,gplearn,A,0.0,0,ok,log((V_DS + V_DS) / (V_GS - V_DS)),8,0.695386,0.510638,1.0,1.0,False,não converge,0.284602,0,0.000000
6,N3,pysr,A,0.0,0,ok,2.006235814654019,1,3.521205,0.605476,1.0,1.0,False,não converge,0.104143,0,0.000000
7,N3,deap,A,0.0,0,ok,log((V_DS + V_DS) / sin(V_DS)),7,8.254037,0.610542,1.0,1.0,False,não converge,0.030472,0,0.000000
8,N3,gplearn,A,0.0,0,ok,log(V_DS),2,0.884342,1.510662,1.0,1.0,False,não converge,0.236850,0,0.000000
9,N4,pysr,A,0.0,0,ok,0.0324618284635074,1,1.005312,0.792194,1.0,1.0,False,não converge,0.221705,0,0.000000


recupera  one_minus_r2_mediana  rmse_log10_mediana  \
nivel motor   condicao                                                       
N1    deap    A              0.0              0.868459            0.491110   
      gplearn A              0.0              0.744881            0.383453   
      pysr    A              0.0              1.020656            0.474479   
N2    deap    A              0.0              1.289586            0.525873   
              B              0.0              2.303226            7.067043   
              C              0.0              2.303226            7.067043   
      gplearn A              0.0              0.695386            0.510638   
      pysr    A              0.0              1.000653            0.575566   
N3    deap    A              0.0              8.254037            0.610542   
      gplearn A              0.0              0.884342            1.510662   
      pysr    A              0.0              3.521205            0.605476   
N4    deap    A              0.0              1.252601            0.654475   
      gplearn A              0.0              1.577093            7.276196   
      pysr    A              0.0              1.005312            0.792194   

                        dist_algebrica_mediana  nos_mediana     tempo_s  \
nivel motor   condicao                                                    
N1    deap    A                            1.0          2.0    0.107167   
      gplearn A                            1.0         35.0    0.234971   
      pysr    A                            1.0          1.0   99.187349   
N2    deap    A                            1.0          3.0    0.029232   
              B                            1.0          5.0  102.927106   
              C                            1.0          5.0   52.337342   
      gplearn A                            1.0          8.0    0.284602   
      pysr    A                            1.0          1.0    2.954868   
N3    deap    A                            1.0          7.0    0.030472   
      gplearn A                            1.0          2.0    0.236850   
      pysr    A                            1.0          1.0    0.104143   
N4    deap    A                            1.0          3.0    0.029262   
      gplearn A                            1.0          3.0    0.229462   
      pysr    A                            1.0          1.0    0.221705   

                        chamadas_llm  
nivel motor   condicao                
N1    deap    A                    0  
      gplearn A                    0  
      pysr    A                    0  
N2    deap    A                    0  
              B                    9  
              C                    9  
      gplearn A                    0  
      pysr    A                    0  
N3    deap    A                    0  
      gplearn A                    0  
      pysr    A                    0  
N4    deap    A                    0  
      gplearn A                    0  
      pysr    A                    0

In [10]:
if len(df) and (df["status"] == "ok").any():
    ok = df[df["status"] == "ok"]
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    marcadores = {"pysr": "o", "deap": "s", "gplearn": "^"}
    cores = {"A": "C0", "B": "C1", "C": "C2"}
    for (motor, cond), g in ok.groupby(["motor", "condicao"]):
        ax.scatter(g["n_nos"], np.maximum(g["one_minus_r2"], 1e-16), marker=marcadores[motor],
                   color=cores[cond], label=f"{motor} / {cond}", alpha=0.8)
    for nid in sorted(ok["nivel"].unique()):
        ax.axvline(cp.loc[nid, "n_nos"], color="grey", lw=0.5, ls=":")
    ax.set(xscale="linear", yscale="log", xlabel="complexidade (nós)", ylabel="1 − R² (teste)",
           title="Erro × complexidade" + (" — FUMAÇA" if FUMACA else ""))
    ax.legend(fontsize=7); fig.tight_layout()
    fig.savefig(DIR_RESULTADOS / f"fig_pareto_{ORCAMENTO}.png", dpi=150); plt.show()

## 7. Exportação

In [11]:
carimbo = time.strftime("%Y%m%d-%H%M%S")
base = DIR_RESULTADOS / f"rs_otft_{ORCAMENTO}_{carimbo}"
df.to_csv(base.with_suffix(".csv"), index=False)
meta = {"orcamento": ORCAMENTO, "fumaca": FUMACA, "niveis": NIVEIS, "motores": MOTORES,
        "condicoes": CONDICOES, "ruidos": RUIDOS, "sementes": SEMENTES, "backend_llm": BACKEND_LLM,
        "criterio": {"max_one_minus_r2": CRITERIO.max_one_minus_r2,
                     "max_tree_distance": CRITERIO.max_tree_distance},
        "niveis_referencia": {n.id: n.expressao for n in escada()},
        "parametros_org1": p.como_dict(),
        "linhas": json.loads(df.to_json(orient="records"))}
base.with_suffix(".json").write_text(json.dumps(meta, ensure_ascii=False, indent=1))
print("exportado:", base.with_suffix(".csv").name, base.with_suffix(".json").name)

exportado: rs_otft_fumaca_20261115-124942.csv rs_otft_fumaca_20261115-124942.json
